In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import sys

sys.path.append("../..")
from src import *

In [ ]:
OUTPUT_DIR = Path("dataset")
(OUTPUT_DIR/"diffuse").mkdir(exist_ok=True, parents=True)
(OUTPUT_DIR/"uv").mkdir(exist_ok=True)
TESTSET_DIR = Path("../dataset/test")
testset = pd.read_json("../dataset/test/metadata.jsonl", orient="records", lines=True)

In [ ]:
ckpt = "stabilityai/stable-diffusion-xl-base-1.0"
controlnet = "trainings/sdxl_16bs_-5lr_2k.safetensors"
# controlnet = "checkpoints/bdsqlsz_controlllite_xl_mlsd_V2.safetensors"
steps = 30
out_dir = Path("outputs") / controlnet.split("/")[-1]
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for _, sample in tqdm(list(testset.iterrows())):
    cmd = [
        "python",
        "sdxl_gen_img.py",
        "--ckpt",
        ckpt,
        "--control_net_lllite_models",
        controlnet,
        "--guide_image_path",
        str((TESTSET_DIR / sample.uv_file_name).resolve()),
        "--prompt",
        sample.caption,
        "--outdir",
        str(out_dir),
        "--use_original_file_name",
        "--W",
        "1024",
        "--H",
        "1024",
        "--bf16",
        "--batch_size",
        "1",
        "--steps",
        str(steps),
    ]
    subprocess.run(cmd, check=True, capture_output=False)

In [ ]:
out_dir = Path("outputs") / "bdsqlsz_controlllite_xl_mlsd_V2.safetensors"
outputs = sorted(out_dir.glob("*.png"), key=lambda f:f.stem.split("_")[1])
for uid, file in zip(testset.uv_file_name, outputs):
    file.rename(file.with_stem(Path(uid).stem))